# 02 · Cleaning & Feature Engineering
**Goal:** Create features that *define* impulse behaviour and classify each transaction.

In [1]:
import pandas as pd, numpy as np, os
df = pd.read_csv('../data/raw/blinkit_dataset (2).csv')
print('Loaded:', df.shape)

Loaded: (13000, 25)


### 1 · Handle Missing Values
`offer_type` has NaN (pandas reads CSV string `None` as null). Fill with `No Offer`.

In [2]:
df['offer_type'] = df['offer_type'].fillna('No Offer')
for c in df.select_dtypes('number').columns:
    df[c] = df[c].fillna(df[c].median())
for c in df.select_dtypes('object').columns:
    df[c] = df[c].fillna(df[c].mode()[0])
print('Remaining nulls:', df.isnull().sum().sum())

Remaining nulls: 0


/var/folders/x7/kph4sypj62b9k8w59g173t1h0000gn/T/ipykernel_84848/3406061738.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes('object').columns:


### 2 · Remove Duplicates

In [3]:
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Removed {before - len(df)} duplicates -> {len(df)} rows')

Removed 0 duplicates -> 13000 rows


### 3 · Standardise Text

In [4]:
for c in df.select_dtypes('object').columns:
    df[c] = df[c].astype(str).str.lower().str.strip()
print('Text standardised.')
df[['category','offer_type','city']].head()

Text standardised.


/var/folders/x7/kph4sypj62b9k8w59g173t1h0000gn/T/ipykernel_84848/3367229405.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes('object').columns:


,category,offer_type,city
0,grocery,no offer,bengaluru
1,dairy,no offer,jaipur
2,personal care,freedelivery,chennai
3,household,no offer,delhi
4,beverages,no offer,pune


## Feature Engineering
Create columns that capture impulse-purchase signals.

In [5]:
df['revenue_inr'] = df['final_price'] * df['sold_quantity']
df['price_per_100g'] = df['final_price'] / (df['weight_g'] / 100)
df['weight_bucket'] = np.where(df['weight_g'] < 300, 'small', np.where(df['weight_g'] <= 700, 'mid', 'bulk'))
df['is_perishable'] = (df['shelf_life_days'] <= 30).astype(int)
impulse_cats = ['snacks','dairy','beverages','bakery','fruits & vegetables']
df['is_impulse_category'] = df['category'].isin(impulse_cats).astype(int)
df['has_offer'] = (df['offer_type'] != 'no offer').astype(int)
print('Features created. New columns:', ['revenue_inr','price_per_100g','weight_bucket','is_perishable','is_impulse_category','has_offer'])

Features created. New columns: ['revenue_inr', 'price_per_100g', 'weight_bucket', 'is_perishable', 'is_impulse_category', 'has_offer']


## Impulse Score & Basket Classification
`impulse_score = 0.30*(is_small) + 0.25*(is_perishable) + 0.25*(is_impulse_category) + 0.20*(has_offer)`

In [6]:
df['is_small'] = (df['weight_bucket'] == 'small').astype(int)
df['impulse_score'] = 0.30*df['is_small'] + 0.25*df['is_perishable'] + 0.25*df['is_impulse_category'] + 0.20*df['has_offer']
df['basket_type'] = np.where(df['impulse_score'] >= 0.6, 'Impulse', np.where(df['impulse_score'] <= 0.4, 'Planned', 'Mixed'))
print('Basket type distribution:')
print(df['basket_type'].value_counts())
print(f'\nImpulse score stats:\n{df["impulse_score"].describe()}')

Basket type distribution:
basket_type
Planned    5117
Mixed      4139
Impulse    3744
Name: count, dtype: int64

Impulse score stats:
count    13000.000000
mean         0.452688
std          0.273635
min          0.000000
25%          0.250000
50%          0.500000
75%          0.700000
max          1.000000
Name: impulse_score, dtype: float64


### Advanced Features

In [7]:
cat_avg = df.groupby('category')['price_per_100g'].transform('mean')
df['impulse_premium_pct'] = ((df['price_per_100g'] - cat_avg) / cat_avg) * 100
city_mean = df.groupby('city')['impulse_score'].transform('mean')
global_mean = df['impulse_score'].mean()
df['city_impulse_index'] = city_mean / global_mean if global_mean > 0 else 1.0
print('Advanced features added.')
print(f'Global impulse score mean: {global_mean:.3f}')

Advanced features added.
Global impulse score mean: 0.453


### 4 · Save Cleaned Dataset

In [8]:
os.makedirs(os.path.dirname('../data/processed/cleaned.csv'), exist_ok=True)
df.to_csv('../data/processed/cleaned.csv', index=False)
print('Saved to ../data/processed/cleaned.csv ->', df.shape)

Saved to ../data/processed/cleaned.csv -> (13000, 36)
